# Comparison with established methods (Tables 10 to 12)

Compares the bounded residual update and the Mahalanobis screening with standard published alternatives on the same
DENTEX tooth crops, the same cached embeddings, the same radiograph-grouped splits (seeds 42, 43, 44) and the same
metrics as the manuscript.

* **Part A** (three splits): prediction update and calibration. Macro-AUC, macro-ECE and the largest change from the
  reference prediction on the test split.
* **Part B** (seed 42): shift screening. DENTEX test crops against Tufts and ADCD crops: AUROC, FPR95, refusal at the
  DENTEX validation threshold, and refusal when 95% of DENTEX test crops are retained.
* **Part C** (seed 42): refusal on DENTEX test crops. Element-wise accuracy of the bounded update as crops are
  deferred by each score.

Every setting is fixed in advance or chosen on the validation split. The test split is only evaluated.

Inputs: cached files only (no radiographs, no encoders, no GPU). Runtime: a few minutes on the CPU.
Output: `_results_point1/baselines_comparison.json` in the DENTEX folder. The run reported in the manuscript is in `results/baselines_comparison.json` of this repository.

In [ ]:
import os, glob, json, time, datetime, platform
import numpy as np, torch, torch.nn as nn, sklearn, scipy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from scipy.special import expit as sig, logsumexp

DEVICE = "cpu"                     # small heads; the CPU with fixed seeds gives the same numbers on every run
torch.set_num_threads(max(1, (os.cpu_count() or 2) - 2))
print("python", platform.python_version(), "| torch", torch.__version__, "| sklearn", sklearn.__version__,
      "| numpy", np.__version__, "| device", DEVICE)

## 1. Paths

In [ ]:
DENTAL     = "/path/to/Dental"     # folder that contains DENTEX/ (and ADCD/)
DENTEX_DIR = os.path.join(DENTAL, "DENTEX/training_data/quadrant-enumeration-disease")
ANN_PATH   = os.path.join(DENTEX_DIR, "train_quadrant_enumeration_disease.json")
CACHE      = os.path.join(DENTEX_DIR, "_results_option1", "emb_cache")
OUT_DIR    = os.path.join(DENTEX_DIR, "_results_point1"); os.makedirs(OUT_DIR, exist_ok=True)

# ADCD embeddings from the ADCD notebook (Dental ViT-L only). If the search finds nothing, set the full path here.
ADCD_EMB = None
if ADCD_EMB is None:
    hits = glob.glob(os.path.join(DENTAL, "**", "_cache", "emb_adcd_dental_vitl.npy"), recursive=True)
    ADCD_EMB = hits[0] if hits else None

# Tufts embeddings: the cache behind the manuscript's Tufts numbers is tufts_box_emb_*.npy; a fresh run of
# 04_tufts_shift_screening.ipynb writes tufts_toothcrop_emb_*.npy instead.
TUFTS_FMT = "tufts_box_emb_{}.npy" if os.path.exists(os.path.join(CACHE, "tufts_box_emb_dental_vitl.npy")) \
            else "tufts_toothcrop_emb_{}.npy"
print("Tufts cache:", TUFTS_FMT.format("<encoder>"))
NEEDED = [ANN_PATH] + [os.path.join(CACHE, f) for f in ("emb_dental_vitl.npy", "emb_biomedclip.npy",
                                                        TUFTS_FMT.format("dental_vitl"), TUFTS_FMT.format("biomedclip"))]
for p in NEEDED:
    assert os.path.exists(p), f"missing: {p}"
print("all DENTEX and Tufts files found")
print("ADCD:", ADCD_EMB if ADCD_EMB else "not found, ADCD rows will be skipped")

## 2. DENTEX tooth labels and splits (same code as the three-split notebook)

In [ ]:
coco = json.load(open(ANN_PATH))
cats3 = sorted(coco["categories_3"], key=lambda c: c["id"])
DX_NAMES = [c["name"] for c in cats3]; NUM_DX = len(DX_NAMES)
id2idx = {c["id"]: i for i, c in enumerate(cats3)}
tooth = {}
for a in coco["annotations"]:
    key = (a["image_id"], a.get("category_id_1", -1), a.get("category_id_2", -1))
    if key not in tooth:
        tooth[key] = {"image_id": a["image_id"], "y": np.zeros(NUM_DX, np.float32)}
    tooth[key]["y"][id2idx[a["category_id_3"]]] = 1.0
teeth = list(tooth.values())
Y_ALL = np.stack([t["y"] for t in teeth])
IMG_OF = np.array([t["image_id"] for t in teeth])
POS = Y_ALL.sum(0).astype(int).tolist()
print(len(teeth), "teeth |", dict(zip(DX_NAMES, POS)))
assert len(teeth) == 3456 and sorted(POS) == sorted([604, 2186, 158, 575]), "labels differ from the manuscript"

def grouped_split(img_of, seed, vf=0.15, tf=0.15):
    ids = sorted(set(img_of.tolist())); rng = np.random.RandomState(seed); rng.shuffle(ids)
    n = len(ids); nte = int(n * tf); nva = int(n * vf)
    te = set(ids[:nte]); va = set(ids[nte:nte + nva]); tr_, va_, te_ = [], [], []
    for i, im in enumerate(img_of):
        (te_ if im in te else va_ if im in va else tr_).append(i)
    return np.array(tr_), np.array(va_), np.array(te_)

tr42, va42, te42 = grouped_split(IMG_OF, 42)
print("seed-42 split:", len(tr42), len(va42), len(te42))
assert (len(tr42), len(va42), len(te42)) == (2416, 511, 529), "seed-42 split differs from the manuscript"

## 3. Cached embeddings

In [ ]:
EMB = {n: np.load(os.path.join(CACHE, f"emb_{n}.npy")).astype(np.float32) for n in ("dental_vitl", "biomedclip")}
EXT = {n: {"Tufts": np.load(os.path.join(CACHE, TUFTS_FMT.format(n))).astype(np.float32)} for n in EMB}
if ADCD_EMB:
    EXT["dental_vitl"]["ADCD"] = np.load(ADCD_EMB).astype(np.float32)
for n in EMB:
    assert len(EMB[n]) == len(teeth), (n, EMB[n].shape)
    print(n, "DENTEX", EMB[n].shape, "|", {k: v.shape for k, v in EXT[n].items()})

## 4. Methods

The reference probe, temperature, clipped residual head and early stopping are copied from the three-split notebook.
The only change is a fixed `torch.manual_seed` per head, so the comparison can be repeated exactly.

* **Probe + Platt / isotonic**: per-condition Platt scaling and isotonic regression fitted on validation probe outputs.
* **Bounded residual (ours)**: reference logit plus residual clipped to ±δ, δ = 1.5.
* **Bounded residual, 5-member ensemble**: five bounded heads, probabilities averaged. The average stays within ±δ.
* **Residual without clip**: the same head with the clip removed.
* **MLP head + temperature**: the same architecture trained from the labels alone, then temperature-scaled on validation.
* **Deep ensemble**: five such MLP heads with different seeds, probabilities averaged (Lakshminarayanan et al., 2017).
* **MC dropout**: the first MLP head with dropout kept on at test time, 30 passes averaged (Gal and Ghahramani, 2016).

In [ ]:
def fit_baseline(X, Yt):
    return [LogisticRegression(max_iter=1000, C=1.0).fit(X, Yt[:, c]) if len(np.unique(Yt[:, c])) > 1 else None
            for c in range(NUM_DX)]
def baseline_logits(pr, X):
    o = np.zeros((len(X), NUM_DX), np.float32)
    for c, p in enumerate(pr): o[:, c] = p.decision_function(X) if p is not None else 0.0
    return o
def fit_temp(logits, y):
    T = nn.Parameter(torch.ones(1)); z = torch.tensor(np.asarray(logits, np.float32)); yt = torch.tensor(y)
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=60); bce = nn.BCEWithLogitsLoss()
    def cl(): opt.zero_grad(); l = bce(z / T.clamp(min=0.05), yt); l.backward(); return l
    opt.step(cl); return float(T.detach().clamp(min=0.05))
def ece(y, p, nb=10, count_zero=False):
    # Manuscript code: bins (lo, hi], so a probability of exactly 0 falls in no bin. Only isotonic regression
    # outputs exact zeros; for it, count_zero=True also reports the value with 0 placed in the first bin.
    b = np.linspace(0, 1, nb + 1); e = 0.0
    for i, (lo, hi) in enumerate(zip(b[:-1], b[1:])):
        m = ((p >= lo) if (i == 0 and count_zero) else (p > lo)) & (p <= hi)
        if m.sum() == 0: continue
        e += (m.sum() / len(p)) * abs(p[m].mean() - y[m].mean())
    return e
def macro_auc(yt, yp):
    v = [roc_auc_score(yt[:, c], yp[:, c]) for c in range(NUM_DX) if len(np.unique(yt[:, c])) == 2]
    return float(np.mean(v)) if v else float("nan")
def macro_ece(yt, yp, count_zero=False):
    return float(np.mean([ece(yt[:, c], yp[:, c], count_zero=count_zero) for c in range(NUM_DX)]))

class EnvHead(nn.Module):
    def __init__(self, d, k, delta):
        super().__init__(); self.delta = delta
        self.net = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, k))
    def residual(self, x): return torch.clamp(self.net(x), -self.delta, self.delta)
    def forward(self, x, b): return b + self.residual(x)

def train_env(Xtr, btr, Ytr, Xva, bva, Yva, delta, epochs=60, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    h = EnvHead(Xtr.shape[1], NUM_DX, delta).to(DEVICE)
    opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=1e-4); crit = nn.BCEWithLogitsLoss()
    dl = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(btr), torch.tensor(Ytr)), batch_size=64, shuffle=True)
    best = 1e9; bs = None
    for _ in range(epochs):
        h.train()
        for x, b, y in dl:
            x, b, y = x.to(DEVICE), b.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); crit(h(x, b), y).backward(); opt.step()
        h.eval()
        with torch.no_grad():
            vl = crit(h(torch.tensor(Xva).to(DEVICE), torch.tensor(bva).to(DEVICE)), torch.tensor(Yva).to(DEVICE)).item()
        if vl < best: best = vl; bs = {k: v.detach().cpu().clone() for k, v in h.state_dict().items()}
    h.load_state_dict(bs); return h

def head_logits(h, X, b):
    h.eval()
    with torch.no_grad():
        return h(torch.tensor(X), torch.tensor(b)).numpy().astype(np.float64)
def mc_logits(h, X, b, passes=30, seed=0):
    torch.manual_seed(seed); h.train()                       # dropout on
    with torch.no_grad():
        Ls = np.stack([h(torch.tensor(X), torch.tensor(b)).numpy().astype(np.float64) for _ in range(passes)])
    h.eval(); return Ls                                       # passes x N x K
def logit_of_mean(Ls):
    # logit of the probability averaged over members (axis 0), computed without leaving log space
    return logsumexp(-np.logaddexp(0, -Ls), axis=0) - logsumexp(-np.logaddexp(0, Ls), axis=0)
def to_logit(p):
    p = np.clip(np.asarray(p, np.float64), 1e-12, 1 - 1e-12); return np.log(p / (1 - p))
def zeros(X): return np.zeros((len(X), NUM_DX), np.float32)
print("methods ready")

## 5. Part A: prediction update and calibration, three splits

In [ ]:
SEEDS = [42, 43, 44]; DELTA = 1.5; N_ENS = 5; NO_CLIP = 1e9
METHODS = ["Probe, no calibration", "Probe + temperature (reference)", "Probe + Platt", "Probe + isotonic",
           "Bounded residual, delta 1.5 (ours)", "Bounded residual, 5-member ensemble", "Residual without clip",
           "MLP head + temperature", "Deep ensemble, 5 MLP heads", "MC dropout, MLP head"]
A = {n: {m: {"auc": [], "ece": [], "max_dlogit": [], "max_dprob": []} for m in METHODS} for n in EMB}
for n in EMB: A[n]["Probe + isotonic"]["ece_zero_counted"] = []
KEEP = {}
t0 = time.time()
for n in EMB:
    X = EMB[n]
    for s in SEEDS:
        tr, va, te = grouped_split(IMG_OF, s)
        Xtr, Xva, Xte = X[tr], X[va], X[te]; Ytr, Yva, Yte = Y_ALL[tr], Y_ALL[va], Y_ALL[te]
        pr = fit_baseline(Xtr, Ytr)
        ztr, zva, zte = baseline_logits(pr, Xtr), baseline_logits(pr, Xva), baseline_logits(pr, Xte)
        T = fit_temp(zva, Yva)
        btr, bva, bte = ztr / T, zva / T, zte / T                   # reference logits (float32)
        ref = bte.astype(np.float64)
        L, P = {}, {}                                              # test logits, or probabilities when no logit exists
        L["Probe, no calibration"] = zte.astype(np.float64)
        L["Probe + temperature (reference)"] = ref
        # Platt and isotonic exactly as in the seed-42 calibration comparison of the single-split notebook
        L["Probe + Platt"] = np.stack([LogisticRegression().fit(zva[:, [c]], Yva[:, c])
                                       .decision_function(zte[:, [c]]) for c in range(NUM_DX)], 1)
        P["Probe + isotonic"] = np.stack([IsotonicRegression(out_of_bounds="clip")
                                          .fit(sig(zva[:, c]), Yva[:, c]).predict(sig(zte[:, c])) for c in range(NUM_DX)], 1)
        env = [train_env(Xtr, btr, Ytr, Xva, bva, Yva, DELTA, seed=1000 * s + m) for m in range(N_ENS)]
        Lenv = [head_logits(h, Xte, bte) for h in env]
        L["Bounded residual, delta 1.5 (ours)"] = Lenv[0]
        L["Bounded residual, 5-member ensemble"] = logit_of_mean(np.stack(Lenv))
        h_nc = train_env(Xtr, btr, Ytr, Xva, bva, Yva, NO_CLIP, seed=1000 * s + 10)
        L["Residual without clip"] = head_logits(h_nc, Xte, bte)
        mlp = [train_env(Xtr, zeros(Xtr), Ytr, Xva, zeros(Xva), Yva, NO_CLIP, seed=1000 * s + 20 + m) for m in range(N_ENS)]
        T_mlp = fit_temp(head_logits(mlp[0], Xva, zeros(Xva)), Yva)
        L["MLP head + temperature"] = head_logits(mlp[0], Xte, zeros(Xte)) / T_mlp
        L["Deep ensemble, 5 MLP heads"] = logit_of_mean(np.stack([head_logits(h, Xte, zeros(Xte)) for h in mlp]))
        L["MC dropout, MLP head"] = logit_of_mean(mc_logits(mlp[0], Xte, zeros(Xte), seed=s))
        for m in METHODS:
            lg = L[m] if m in L else to_logit(P[m])
            p = sig(L[m]) if m in L else P[m]
            A[n][m]["auc"].append(macro_auc(Yte, p)); A[n][m]["ece"].append(macro_ece(Yte, p))
            A[n][m]["max_dlogit"].append(float(np.max(np.abs(lg - ref))))
            A[n][m]["max_dprob"].append(float(np.max(np.abs(p - sig(ref)))))
        A[n]["Probe + isotonic"]["ece_zero_counted"].append(macro_ece(Yte, P["Probe + isotonic"], count_zero=True))
        if s == 42:
            KEEP[n] = dict(tr=tr, va=va, te=te, pr=pr, T=T, env=env, mlp=mlp, Yte=Yte, p_env=sig(Lenv[0]))
        print(f"{n} seed {s} done ({time.time() - t0:.0f} s)")

def fmt(v): return f"{np.mean(v):.3f} ± {np.std(v):.3f}"
for n in EMB:
    print(f"\n=== Part A: {n} (test split, mean ± sd over seeds {SEEDS}) ===")
    print(f"{'method':40s} {'macro-AUC':>15s} {'macro-ECE':>15s} {'max |Δ logit|':>14s} {'max |Δ p|':>10s}")
    for m in METHODS:
        r = A[n][m]
        print(f"{m:40s} {fmt(r['auc']):>15s} {fmt(r['ece']):>15s} {max(r['max_dlogit']):14.3f} {max(r['max_dprob']):10.3f}")
    ok = max(A[n]["Bounded residual, delta 1.5 (ours)"]["max_dlogit"]) <= DELTA + 1e-4
    print("bound holds for ours on every split:", ok)
print("\nΔ is measured against the temperature-scaled reference probe. Ensembles and MC dropout use the logit of the"
      " averaged probability. Isotonic regression has no logit output; its Δ logit uses probabilities clipped to"
      " [1e-12, 1 - 1e-12], so read its Δ p instead.")

## 6. Part B: shift screening, seed-42 split

Every score is fitted on the DENTEX training split; higher means less familiar.

* **Mahalanobis (ours)**: one Gaussian over all training embeddings, ridge 10⁻².
* **Class-conditional Mahalanobis** (Lee et al., 2018): one mean per condition from the training teeth carrying it, one
  shared covariance, minimum over conditions.
* **Relative Mahalanobis** (Ren et al., 2021): class-conditional distance minus the single-Gaussian distance.
* **k-NN distance** (Sun et al., 2022): distance to the 50th nearest training embedding after L2 normalisation.
* **Max sigmoid, energy, ODIN (T = 1000)**: the logit summaries already in Table 7, on the uncalibrated probe logits.
* **Deep ensemble / MC dropout**: mean variance of the predicted probabilities across members or passes.

In [ ]:
def auroc(d_in, d_ood):            # same function as the manuscript's Table 7
    s = np.concatenate([d_ood, d_in]); y = np.concatenate([np.ones(len(d_ood)), np.zeros(len(d_in))])
    o = np.argsort(s); r = np.empty_like(o, float); r[o] = np.arange(1, len(s) + 1)
    npos, nneg = int(y.sum()), int((y == 0).sum())
    return float((r[y == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg))
def fpr95(d_in, d_ood):
    return float((d_in >= np.quantile(d_ood, 0.05)).mean())
def maha_fn(mu, ci):
    return lambda X: (((X - mu) @ ci) * (X - mu)).sum(1)
K_NN = 50

def make_scores(n):
    k = KEEP[n]; Xtr = EMB[n][k["tr"]]; Ytr = Y_ALL[k["tr"]]; d = Xtr.shape[1]
    mu0 = Xtr.mean(0); ci0 = np.linalg.inv(np.cov(Xtr.T) + np.eye(d) * 1e-2); m0 = maha_fn(mu0, ci0)
    mus = [Xtr[Ytr[:, c] == 1].mean(0) for c in range(NUM_DX)]
    R = np.concatenate([Xtr[Ytr[:, c] == 1] - mus[c] for c in range(NUM_DX)])
    cic = np.linalg.inv(np.cov(R.T) + np.eye(d) * 1e-2)
    def mc(X): return np.stack([maha_fn(m, cic)(X) for m in mus], 1)
    B = Xtr / np.linalg.norm(Xtr, axis=1, keepdims=True)
    def knn(X):
        A_ = X / np.linalg.norm(X, axis=1, keepdims=True); out = []
        for i in range(0, len(A_), 4096):
            S = A_[i:i + 4096] @ B.T
            sk = -np.partition(-S, K_NN - 1, axis=1)[:, K_NN - 1]
            out.append(np.sqrt(np.clip(2 - 2 * sk, 0, None)))
        return np.concatenate(out)
    pr = k["pr"]
    def lg(X): return baseline_logits(pr, X).astype(np.float64)
    def ens_var(X): return np.stack([sig(head_logits(h, X, zeros(X))) for h in k["mlp"]]).var(0).mean(1)
    def mcd_var(X): return sig(mc_logits(k["mlp"][0], X, zeros(X), seed=42)).var(0).mean(1)
    return {
        "Mahalanobis (ours)": m0,
        "Class-conditional Mahalanobis": lambda X: mc(X).min(1),
        "Relative Mahalanobis": lambda X: (mc(X) - m0(X)[:, None]).min(1),
        "k-NN distance (k=50)": knn,
        "Max sigmoid (proxy)": lambda X: -sig(lg(X)).max(1),
        "Energy (proxy)": lambda X: -logsumexp(lg(X), axis=1),
        "ODIN, T=1000 (proxy)": lambda X: -sig(lg(X) / 1000).max(1),
        "Deep ensemble variance": ens_var,
        "MC dropout variance": mcd_var,
    }

B_RES = {}; SCORES_TE = {}
for n in EMB:
    k = KEEP[n]; Xva, Xte = EMB[n][k["va"]], EMB[n][k["te"]]
    B_RES[n] = {}; SCORES_TE[n] = {}
    for sname, f in make_scores(n).items():
        s_va, s_te = f(Xva), f(Xte); SCORES_TE[n][sname] = s_te
        thr_va, thr_te = np.quantile(s_va, 0.95), np.quantile(s_te, 0.95)
        row = {"dentex_test_refused_at_val_threshold": float((s_te > thr_va).mean())}
        for ename, Xe in EXT[n].items():
            s_e = f(Xe)
            row[ename] = {"auroc": auroc(s_te, s_e), "fpr95": fpr95(s_te, s_e),
                          "refused_at_val_threshold": float((s_e > thr_va).mean()),
                          "refused_at_95pct_test_retention": float((s_e > thr_te).mean())}
        B_RES[n][sname] = row
    for ename in EXT[n]:
        print(f"\n=== Part B: {n}, DENTEX test vs {ename} (seed 42) ===")
        print(f"{'score':32s} {'AUROC':>7s} {'FPR95':>7s} {'refused, val thr':>17s} {'refused, 95% test':>18s} {'DENTEX test, val thr':>21s}")
        for sname, row in B_RES[n].items():
            e = row[ename]
            print(f"{sname:32s} {e['auroc']:7.3f} {e['fpr95']:7.3f} {e['refused_at_val_threshold']:17.1%} "
                  f"{e['refused_at_95pct_test_retention']:18.1%} {row['dentex_test_refused_at_val_threshold']:21.1%}")

## 7. Part C: refusal on DENTEX test crops, seed 42

Predictions are those of the bounded update (threshold 0.5, element-wise accuracy over the four conditions, as in
Fig. 4). Teeth are deferred in order of each score. AURC is the mean error over all coverage levels (lower is better).
"Confidence of the update" is the selective-classification baseline of Geifman and El-Yaniv (2017): the mean over
conditions of max(p, 1 − p).

In [ ]:
def risk_coverage(correct, unc):
    order = np.argsort(unc, kind="stable")
    acc = np.cumsum(correct[order]) / np.arange(1, len(correct) + 1)
    at = lambda c: float(acc[int(np.ceil(c * len(correct))) - 1])
    return {"aurc": float((1 - acc).mean()), "acc_at_30": at(0.3), "acc_at_50": at(0.5), "acc_at_70": at(0.7),
            "acc_at_90": at(0.9), "acc_at_100": at(1.0)}
C_RES = {}
for n in EMB:
    k = KEEP[n]; p = k["p_env"]; correct = ((p >= 0.5) == (k["Yte"] == 1)).mean(1)
    U = dict(SCORES_TE[n])
    U["Confidence of the update"] = -np.maximum(p, 1 - p).mean(1)
    U["Oracle (best possible order)"] = -correct
    C_RES[n] = {u: risk_coverage(correct, s) for u, s in U.items()}
    print(f"\n=== Part C: {n}, refusal on DENTEX test (seed 42) ===")
    print(f"{'deferral score':32s} {'AURC':>7s} {'acc@30%':>8s} {'acc@50%':>8s} {'acc@70%':>8s} {'acc@90%':>8s} {'acc@100%':>9s}")
    for u, r in C_RES[n].items():
        print(f"{u:32s} {r['aurc']:7.3f} {r['acc_at_30']:8.3f} {r['acc_at_50']:8.3f} {r['acc_at_70']:8.3f} "
              f"{r['acc_at_90']:8.3f} {r['acc_at_100']:9.3f}")

## 8. Check against the manuscript, then save

The probe, temperature, Mahalanobis and logit-summary numbers do not depend on random initialisation, so they should
match the manuscript. The residual-head numbers can differ slightly, because the original runs did not fix the
head seeds.

In [ ]:
PAPER = {"dental_vitl": {"ref_auc": 0.849, "ref_ece": 0.040, "maha_auroc": 0.968, "maha_fpr95": 0.185,
                         "maha_tufts_95": 0.847, "msp": 0.775, "energy": 0.786, "odin": 0.775, "adcd_val": 0.33,
                         "s42_ece_uncal": 0.038, "s42_ece_temp": 0.040, "s42_ece_platt": 0.033, "s42_ece_iso": 0.02846},
         "biomedclip":  {"ref_auc": 0.790, "ref_ece": 0.036, "maha_auroc": 0.951, "maha_fpr95": 0.306,
                         "maha_tufts_95": 0.782, "msp": 0.553, "energy": 0.553, "odin": 0.553}}
CHECK = {}
for n in EMB:
    a, b, pp = A[n], B_RES[n], PAPER[n]
    got = {"ref_auc": np.mean(a["Probe + temperature (reference)"]["auc"]),
           "ref_ece": np.mean(a["Probe + temperature (reference)"]["ece"]),
           "maha_auroc": b["Mahalanobis (ours)"]["Tufts"]["auroc"], "maha_fpr95": b["Mahalanobis (ours)"]["Tufts"]["fpr95"],
           "maha_tufts_95": b["Mahalanobis (ours)"]["Tufts"]["refused_at_95pct_test_retention"],
           "msp": b["Max sigmoid (proxy)"]["Tufts"]["auroc"], "energy": b["Energy (proxy)"]["Tufts"]["auroc"],
           "odin": b["ODIN, T=1000 (proxy)"]["Tufts"]["auroc"],
           "s42_ece_uncal": a["Probe, no calibration"]["ece"][0], "s42_ece_temp": a["Probe + temperature (reference)"]["ece"][0],
           "s42_ece_platt": a["Probe + Platt"]["ece"][0], "s42_ece_iso": a["Probe + isotonic"]["ece"][0]}
    if "ADCD" in EXT[n]:
        got["adcd_val"] = b["Mahalanobis (ours)"]["ADCD"]["refused_at_val_threshold"]
    CHECK[n] = {key: {"manuscript": pp[key], "now": float(v)} for key, v in got.items() if key in pp}
    print(f"\n{n}:")
    for key, v in CHECK[n].items():
        print(f"  {key:14s} manuscript {v['manuscript']:.4f} | now {v['now']:.4f}")

res = {"version": "baselines_comparison_v1", "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
       "software": {"python": platform.python_version(), "torch": torch.__version__, "sklearn": sklearn.__version__,
                    "numpy": np.__version__, "scipy": scipy.__version__, "device": DEVICE},
       "files": {"annotations": ANN_PATH, "cache": CACHE, "adcd": ADCD_EMB},
       "settings": {"seeds": SEEDS, "delta": DELTA, "n_ensemble": N_ENS, "k_nn": K_NN, "mc_passes": 30,
                    "head_seed_rule": "1000*split_seed + member (bounded), +10 (no clip), +20+member (MLP)"},
       "classes": DX_NAMES, "positives": POS,
       "split_sizes": {str(s): [len(x) for x in grouped_split(IMG_OF, s)] for s in SEEDS},
       "external_sizes": {n: {e: int(len(v)) for e, v in EXT[n].items()} for n in EMB},
       "part_a": A, "part_b": B_RES, "part_c": C_RES, "check_vs_manuscript": CHECK}
path = os.path.join(OUT_DIR, "baselines_comparison.json")
json.dump(res, open(path, "w"), indent=1)
print("\nSaved", path, "")